# Assignment: Bounding-Box Regression for Object Detection

In this assignment, we study object detection as a regression problem.

You are given proposal boxes that already have some objectness: each proposal is assumed to roughly enclose an object. Your task is not to generate proposals. Your task is to learn a regression head that improves a proposal box so that it better matches the ground-truth box.

We compare two regression formulations:

1. absolute box regression
2. R-CNN-style relative box regression

The central question is:

Which regression target is easier to learn, and which gives better box localization after decoding?

## Box representation

We represent a box by its center, width, and height:

$$
B = (x, y, w, h)
$$

where \(x,y\) are the center coordinates and \(w,h\) are positive side lengths.

Each sample contains:

$$
P_i = (x_i^p, y_i^p, w_i^p, h_i^p)
$$

where \(P_i\) is a given proposal box, and

$$
B_i^\ast = (x_i^\ast, y_i^\ast, w_i^\ast, h_i^\ast)
$$

where \(B_i^\ast\) is the ground-truth box.

The proposal is assumed to have objectness, meaning it already roughly encloses the object. The regressor only needs to refine the proposal.

## Absolute regression target

The first approach directly predicts the ground-truth box:

$$
y_i^{abs} =
(x_i^\ast, y_i^\ast, w_i^\ast, h_i^\ast)
$$

This is simple, but it asks the model to predict absolute image coordinates.

## R-CNN-style regression target

The second approach predicts a correction relative to the proposal:

$$
t_x^\ast = \frac{x^\ast - x^p}{w^p}
$$

$$
t_y^\ast = \frac{y^\ast - y^p}{h^p}
$$

$$
t_w^\ast = \log\left(\frac{w^\ast}{w^p}\right)
$$

$$
t_h^\ast = \log\left(\frac{h^\ast}{h^p}\right)
$$

The target is

$$
t^\ast = (t_x^\ast, t_y^\ast, t_w^\ast, t_h^\ast).
$$

The inverse transformation decodes a predicted correction back into a box:

$$
\hat{x} = \hat{t}_x w^p + x^p
$$

$$
\hat{y} = \hat{t}_y h^p + y^p
$$

$$
\hat{w} = w^p \exp(\hat{t}_w)
$$

$$
\hat{h} = h^p \exp(\hat{t}_h)
$$

This target is relative to the proposal. Translation is normalized by proposal size, and scale change is modeled in log-space.

In [ ]:
# Required packages:
# pip install numpy matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## Synthetic dataset generator

This generator creates a dataset without explicit images.

Each proposal already roughly encloses an object. A small feature vector describes the objectness pattern inside the proposal. The feature vector is synthetic, but it plays the role of the information available to a regression head.

The generator returns:

- proposal boxes \(P\)
- ground-truth boxes \(B^\ast\)
- synthetic objectness features \(X\)
- absolute regression targets
- R-CNN relative regression targets

In [ ]:
def generate_objectness_regression_dataset(
    n_samples=5000,
    image_size=256,
    noise_std=0.02,
    random_state=7,
):
    rng = np.random.default_rng(random_state)

    # Given proposal boxes P = (x_p, y_p, w_p, h_p).
    # These proposals already have objectness and roughly enclose an object.
    x_p = rng.uniform(40, image_size - 40, n_samples)
    y_p = rng.uniform(40, image_size - 40, n_samples)
    w_p = rng.uniform(25, 90, n_samples)
    h_p = rng.uniform(25, 90, n_samples)

    proposals = np.column_stack([x_p, y_p, w_p, h_p])

    # Synthetic objectness features.
    # Interpret these as compact measurements of what is inside the proposal.
    o1 = rng.uniform(-2, 2, n_samples)
    o2 = rng.uniform(-2, 2, n_samples)

    # True relative corrections from proposal to object box.
    # The correction depends nonlinearly on objectness features.
    t_x = 0.24 * np.sin(o1) + 0.05 * o2
    t_y = 0.20 * np.cos(o2) - 0.05 * o1
    t_w = 0.10 * o1**2 - 0.04 * o2
    t_h = -0.08 * o2**2 + 0.06 * o1 * o2

    t_clean = np.column_stack([t_x, t_y, t_w, t_h])
    t = t_clean + rng.normal(0.0, noise_std, size=t_clean.shape)

    # Decode the relative correction to obtain the ground-truth box.
    x_star = t[:, 0] * w_p + x_p
    y_star = t[:, 1] * h_p + y_p
    w_star = w_p * np.exp(t[:, 2])
    h_star = h_p * np.exp(t[:, 3])

    gt_boxes = np.column_stack([x_star, y_star, w_star, h_star])

    # Input features available to the regressor.
    # Proposal geometry is included because the regression head is proposal-conditioned.
    X_raw = np.column_stack([
        o1,
        o2,
        x_p / image_size,
        y_p / image_size,
        w_p / image_size,
        h_p / image_size,
    ])

    # Optional nonlinear feature expansion.
    # A simple linear regressor on this feature map can fit nonlinear relationships.
    X_phi = np.column_stack([
        o1,
        o2,
        o1**2,
        o2**2,
        o1 * o2,
        np.sin(o1),
        np.cos(o2),
        x_p / image_size,
        y_p / image_size,
        w_p / image_size,
        h_p / image_size,
    ])

    y_abs = gt_boxes
    y_rcnn = t

    return {
        "X_raw": X_raw,
        "X_phi": X_phi,
        "proposals": proposals,
        "gt_boxes": gt_boxes,
        "y_abs": y_abs,
        "y_rcnn": y_rcnn,
        "image_size": image_size,
    }


data = generate_objectness_regression_dataset()
print(data.keys())
print("X_raw shape:", data["X_raw"].shape)
print("X_phi shape:", data["X_phi"].shape)
print("Proposal shape:", data["proposals"].shape)
print("Ground-truth box shape:", data["gt_boxes"].shape)

## Visualization of proposal and ground-truth pairs

The dashed boxes are proposals. The solid boxes are ground-truth boxes. The small line segment connects each proposal center to the corresponding ground-truth center.

This plot shows what the regression head is expected to learn: a geometric correction from proposal to ground truth.

In [ ]:
def draw_box(ax, box, linestyle="-", linewidth=2.0, label=None):
    x_c, y_c, w, h = box

    x_min = x_c - w / 2
    y_min = y_c - h / 2

    rectangle = plt.Rectangle(
        (x_min, y_min),
        w,
        h,
        fill=False,
        linestyle=linestyle,
        linewidth=linewidth,
        label=label,
    )
    ax.add_patch(rectangle)


def plot_proposal_ground_truth_pairs(data, n_pairs=20):
    proposals = data["proposals"][:n_pairs]
    gt_boxes = data["gt_boxes"][:n_pairs]
    image_size = data["image_size"]

    fig, ax = plt.subplots(figsize=(8, 8))

    for i in range(n_pairs):
        draw_box(
            ax,
            proposals[i],
            linestyle="--",
            linewidth=1.5,
            label="proposal" if i == 0 else None,
        )

        draw_box(
            ax,
            gt_boxes[i],
            linestyle="-",
            linewidth=2.0,
            label="ground truth" if i == 0 else None,
        )

        ax.plot(
            [proposals[i, 0], gt_boxes[i, 0]],
            [proposals[i, 1], gt_boxes[i, 1]],
            linewidth=0.8,
        )

    ax.set_xlim(0, image_size)
    ax.set_ylim(0, image_size)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_xlabel("x coordinate")
    ax.set_ylabel("y coordinate")
    ax.set_title("Proposal Boxes and Ground-Truth Boxes")
    ax.legend()
    plt.show()


plot_proposal_ground_truth_pairs(data, n_pairs=20)

## Helper functions for decoding and IoU

To compare the two regressors fairly, both predictions must be converted into boxes. Then we evaluate box overlap with intersection-over-union:

$$
IoU(A,B)=\frac{|A \cap B|}{|A \cup B|}
$$

A higher IoU means better localization.

In [ ]:
def decode_rcnn_targets(proposals, t_pred):
    x_p, y_p, w_p, h_p = proposals.T

    x = t_pred[:, 0] * w_p + x_p
    y = t_pred[:, 1] * h_p + y_p
    w = w_p * np.exp(t_pred[:, 2])
    h = h_p * np.exp(t_pred[:, 3])

    return np.column_stack([x, y, w, h])


def center_to_corners(boxes):
    x, y, w, h = boxes.T

    x_min = x - w / 2
    y_min = y - h / 2
    x_max = x + w / 2
    y_max = y + h / 2

    return np.column_stack([x_min, y_min, x_max, y_max])


def box_iou_center_format(boxes_a, boxes_b):
    a = center_to_corners(boxes_a)
    b = center_to_corners(boxes_b)

    x_min_inter = np.maximum(a[:, 0], b[:, 0])
    y_min_inter = np.maximum(a[:, 1], b[:, 1])
    x_max_inter = np.minimum(a[:, 2], b[:, 2])
    y_max_inter = np.minimum(a[:, 3], b[:, 3])

    inter_w = np.maximum(0.0, x_max_inter - x_min_inter)
    inter_h = np.maximum(0.0, y_max_inter - y_min_inter)
    intersection = inter_w * inter_h

    area_a = np.maximum(0.0, a[:, 2] - a[:, 0]) * np.maximum(0.0, a[:, 3] - a[:, 1])
    area_b = np.maximum(0.0, b[:, 2] - b[:, 0]) * np.maximum(0.0, b[:, 3] - b[:, 1])

    union = area_a + area_b - intersection
    return intersection / np.maximum(union, 1e-8)

## Task 1: Train an absolute box regressor

Train a regression model that predicts

$$
(x^\ast,y^\ast,w^\ast,h^\ast)
$$

directly from the objectness/proposal features.

Use either `X_raw` or `X_phi`. Start with `X_phi` so the linear model has access to nonlinear feature transformations.

In [ ]:
X = data["X_phi"]
y_abs = data["y_abs"]
y_rcnn = data["y_rcnn"]
proposals = data["proposals"]
gt_boxes = data["gt_boxes"]

(
    X_train,
    X_test,
    y_abs_train,
    y_abs_test,
    y_rcnn_train,
    y_rcnn_test,
    proposals_train,
    proposals_test,
    gt_train,
    gt_test,
) = train_test_split(
    X,
    y_abs,
    y_rcnn,
    proposals,
    gt_boxes,
    test_size=0.25,
    random_state=0,
)

absolute_model = LinearRegression()
absolute_model.fit(X_train, y_abs_train)

abs_pred_boxes = absolute_model.predict(X_test)

abs_target_mse = mean_squared_error(y_abs_test, abs_pred_boxes)
abs_iou = box_iou_center_format(abs_pred_boxes, gt_test)

print("Absolute regressor target MSE:", abs_target_mse)
print("Absolute regressor mean IoU:  ", abs_iou.mean())

## Task 2: Train an R-CNN-style relative regressor

Train a regression model that predicts

$$
(t_x^\ast,t_y^\ast,t_w^\ast,t_h^\ast)
$$

from the same input features.

Then decode the predicted targets into boxes using the proposal boxes.

In [ ]:
rcnn_model = LinearRegression()
rcnn_model.fit(X_train, y_rcnn_train)

rcnn_pred_targets = rcnn_model.predict(X_test)
rcnn_pred_boxes = decode_rcnn_targets(proposals_test, rcnn_pred_targets)

rcnn_target_mse = mean_squared_error(y_rcnn_test, rcnn_pred_targets)
rcnn_iou = box_iou_center_format(rcnn_pred_boxes, gt_test)

print("R-CNN regressor target MSE:", rcnn_target_mse)
print("R-CNN regressor mean IoU:  ", rcnn_iou.mean())

## Visual comparison of predictions

The next plot compares the proposal, ground-truth box, absolute-regression prediction, and R-CNN-style prediction for a few test examples.

In [ ]:
def plot_prediction_comparison(
    proposals,
    gt_boxes,
    abs_pred_boxes,
    rcnn_pred_boxes,
    n_examples=8,
):
    n_examples = min(n_examples, len(gt_boxes))

    fig, axes = plt.subplots(1, n_examples, figsize=(4 * n_examples, 4))

    if n_examples == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        draw_box(ax, proposals[i], linestyle="--", linewidth=1.2, label="proposal")
        draw_box(ax, gt_boxes[i], linestyle="-", linewidth=2.0, label="ground truth")
        draw_box(ax, abs_pred_boxes[i], linestyle=":", linewidth=2.0, label="absolute pred")
        draw_box(ax, rcnn_pred_boxes[i], linestyle="-.", linewidth=2.0, label="rcnn pred")

        x_values = [
            proposals[i, 0],
            gt_boxes[i, 0],
            abs_pred_boxes[i, 0],
            rcnn_pred_boxes[i, 0],
        ]
        y_values = [
            proposals[i, 1],
            gt_boxes[i, 1],
            abs_pred_boxes[i, 1],
            rcnn_pred_boxes[i, 1],
        ]

        ax.set_xlim(min(x_values) - 80, max(x_values) + 80)
        ax.set_ylim(min(y_values) - 80, max(y_values) + 80)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.set_title(f"example {i}")

    axes[0].legend(loc="upper right")
    plt.show()


plot_prediction_comparison(
    proposals_test,
    gt_test,
    abs_pred_boxes,
    rcnn_pred_boxes,
    n_examples=6,
)

## Task 3: Compare target MSE and decoded-box IoU

Fill in the table below after running both regressors.

| Model | Target predicted | Target MSE | Mean IoU |
|---|---:|---:|---:|
| Absolute regression | \((x^\ast,y^\ast,w^\ast,h^\ast)\) | | |
| R-CNN relative regression | \((t_x^\ast,t_y^\ast,t_w^\ast,t_h^\ast)\) | | |

Answer the following questions:

1. Which model has lower target MSE?
2. Which model has higher mean IoU after decoding?
3. Are target MSE and IoU always aligned?
4. Why might the relative parameterization be easier to learn?
5. What role does proposal size play in normalizing translation errors?
6. Why does using \(\log(w^\ast/w^p)\) and \(\log(h^\ast/h^p)\) make sense for scale?

## Task 4: Compare raw features and nonlinear features

Repeat the experiment twice:

1. using `X_raw`
2. using `X_phi`

Report results for both absolute regression and R-CNN relative regression.

The point is to test two separate ideas:

- the choice of regression target
- the choice of feature representation

A linear regressor can fit nonlinear relationships if the input feature map already contains nonlinear transformations.

In [ ]:
def run_experiment_with_features(X, data):
    y_abs = data["y_abs"]
    y_rcnn = data["y_rcnn"]
    proposals = data["proposals"]
    gt_boxes = data["gt_boxes"]

    (
        X_train,
        X_test,
        y_abs_train,
        y_abs_test,
        y_rcnn_train,
        y_rcnn_test,
        proposals_train,
        proposals_test,
        gt_train,
        gt_test,
    ) = train_test_split(
        X,
        y_abs,
        y_rcnn,
        proposals,
        gt_boxes,
        test_size=0.25,
        random_state=0,
    )

    absolute_model = LinearRegression()
    absolute_model.fit(X_train, y_abs_train)
    abs_pred_boxes = absolute_model.predict(X_test)

    rcnn_model = LinearRegression()
    rcnn_model.fit(X_train, y_rcnn_train)
    rcnn_pred_targets = rcnn_model.predict(X_test)
    rcnn_pred_boxes = decode_rcnn_targets(proposals_test, rcnn_pred_targets)

    results = {
        "absolute_target_mse": mean_squared_error(y_abs_test, abs_pred_boxes),
        "absolute_mean_iou": box_iou_center_format(abs_pred_boxes, gt_test).mean(),
        "rcnn_target_mse": mean_squared_error(y_rcnn_test, rcnn_pred_targets),
        "rcnn_mean_iou": box_iou_center_format(rcnn_pred_boxes, gt_test).mean(),
    }

    return results


results_raw = run_experiment_with_features(data["X_raw"], data)
results_phi = run_experiment_with_features(data["X_phi"], data)

print("Results with raw features:")
for key, value in results_raw.items():
    print(f"  {key}: {value:.6f}")

print()
print("Results with nonlinear features:")
for key, value in results_phi.items():
    print(f"  {key}: {value:.6f}")

## Interpretation

The R-CNN-style target is often preferable because it turns box prediction into residual correction.

The proposal already contains objectness and approximately encloses the object. Therefore, the model should not have to learn global image coordinates from scratch. It should learn how to adjust the proposal.

The relative target has three advantages:

1. It predicts residuals rather than absolute coordinates.
2. It normalizes translation by proposal width and height.
3. It represents multiplicative scale change as additive log-scale change.

This usually gives a better-conditioned regression problem.

## Optional extension

Try replacing `LinearRegression` with `Ridge`:

```python
model = Ridge(alpha=1.0)
```

Then compare:

- absolute regression with raw features
- absolute regression with nonlinear features
- R-CNN regression with raw features
- R-CNN regression with nonlinear features

Write a short conclusion explaining which design choice matters most for this synthetic dataset.